# 04. Prueba de la unión de ediciones de CAMMESA

Prueba de la lectura de las fechas en el nombre de los archivos y de la unión de las cuatro ediciones descargadas (tarea 5 del backlog, HU-01). Cada edición de CAMMESA puede corregir meses publicados en ediciones anteriores, por lo que la serie se arma conservando, para cada mes, el valor de la edición más nueva.

In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
from pathlib import Path
from carga_cammesa import extraer_fechas
from carga_cammesa import union_filtro
from carga_cammesa import csv_tabla

## 1. Fechas en el nombre del archivo

El nombre de cada archivo sigue la convención `CAMMESA_MEM_ReqMax_aaaa-mm_aaaa-mm-dd`: la primera fecha es la edición de CAMMESA y la segunda, el día en que se descargó. `extraer_fechas` las separa a partir de los guiones bajos.

In [3]:
ruta = Path("../data/raw/cammesa/CAMMESA_MEM_ReqMax_2026-09_2026-10-20.xlsx")
print("stem: ",ruta.stem)
print("split: ",ruta.stem.split("_"))

stem:  CAMMESA_MEM_ReqMax_2026-09_2026-10-20
split:  ['CAMMESA', 'MEM', 'ReqMax', '2026-09', '2026-10-20']


In [4]:
edicion, descarga = extraer_fechas(ruta)
edicion, descarga

('2026-09', '2026-10-20')

## 2. Unión de las ediciones

`union_filtro` apila las cuatro ediciones y conserva un registro por mes. Si un mes aparece en varias ediciones, se conserva la más nueva, y si empatan, la descarga más reciente.

In [5]:
rutas = sorted(Path("../data/raw/cammesa").glob("CAMMESA_MEM_ReqMax_*.xlsx"))
tabla = union_filtro(rutas)

In [6]:
tabla.info()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   MES                        93 non-null     datetime64[us]
 1   MAX_REAL_MW                93 non-null     float64       
 2   POTENCIA_DECLARADA_MW      93 non-null     float64       
 3   EDICION                    93 non-null     str           
 4   DESCARGA                   93 non-null     str           
 5   REVISADO                   93 non-null     bool          
 6   POSIBLE_COPIA_DECLARACION  93 non-null     bool          
dtypes: bool(2), datetime64[us](1), float64(2), str(2)
memory usage: 5.5 KB


In [7]:
tabla["EDICION"].value_counts()

EDICION
2026-09    45
2024-12    24
2022-12    12
2023-12    12
Name: count, dtype: int64

In [8]:
tabla["MES"].nunique()

93

In [9]:
ruta_salida = Path("../data/processed/CAMMESA_CTRELEUW_ReqMax.csv")
csv_tabla(tabla,ruta_salida)

## 3. Control del nombre de los archivos

Un archivo con un nombre que no respeta la convención detiene la carga con un error que explica el formato esperado, evitando leer fechas equivocadas.

In [10]:
try:
    extraer_fechas(Path("../data/raw/cammesa/CAMMESA_MEM_ReqMax_2026_09_2026_10_20.xlsx"))
except ValueError as error:
    print("Error esperado:\n", error)

Error esperado:
 Nombre inválido: CAMMESA_MEM_ReqMax_2026_09_2026_10_20.xlsx, se espera CAMMESA_MEM_ReqMax_aaaa-mm_aaaa-mm-dd 
 siendo la primera la fecha de EDICIÓN y la segunda la fecha de DESCARGA


## Conclusión

- La serie unida tiene 93 meses, de enero 2019 a septiembre 2026, sin repetidos.
- La edición 2026-09 aporta 45 meses (2023 a 2026), la 2024-12 aporta 24 (2021 y 2022) y la 2022-12, otros 24 (2019 y 2020).
- La edición 2023-12 no aporta ningún mes a la serie final: todos los que cubre aparecen también en una edición más nueva. Se conserva porque permite detectar los meses que CAMMESA corrigió entre ediciones (notebook 05).